In [1]:
import os
import random
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [3]:
X_train = np.load(
    "X_train_20step.npy",
    mmap_mode="r"
)

y_train = np.load(
    "y_train_20step.npy",
    mmap_mode="r"
)

X_val = np.load(
    "X_val_20step.npy",
    mmap_mode="r"
)

y_val = np.load(
    "y_val_20step.npy",
    mmap_mode="r"
)

X_test = np.load(
    "X_test_20step.npy",
    mmap_mode="r"
)

y_test = np.load(
    "y_test_20step.npy",
    mmap_mode="r"
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (44325, 20, 105) (44325,)
Validation: (9655, 20, 105) (9655,)
Test: (10075, 20, 105) (10075,)


In [4]:
class RULSequenceDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        X_item = torch.tensor(
            self.X[idx],
            dtype=torch.float32
        )

        y_item = torch.tensor(
            self.y[idx],
            dtype=torch.float32
        )

        return X_item, y_item

In [5]:
train_dataset = RULSequenceDataset(
    X_train,
    y_train
)

val_dataset = RULSequenceDataset(
    X_val,
    y_val
)

test_dataset = RULSequenceDataset(
    X_test,
    y_test
)

print("Datasets created.")

Datasets created.


In [6]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 347
Validation batches: 76
Test batches: 79


In [7]:
y_mean = float(np.mean(y_train))
y_std = float(np.std(y_train))

print("Training RUL mean:", y_mean)
print("Training RUL std:", y_std)

Training RUL mean: 88.34149169921875
Training RUL std: 70.04686737060547


In [8]:
class LSTMRegressor(nn.Module):

    def __init__(
        self,
        input_size=105,
        hidden_size=64,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        # x:
        # (batch, 20, 105)

        output, (hidden, cell) = self.lstm(x)

        # Use the final timestep representation
        last_output = output[:, -1, :]

        prediction = self.regressor(
            last_output
        )

        return prediction.squeeze(1)

In [9]:
model = LSTMRegressor(
    input_size=105,
    hidden_size=64,
    num_layers=2,
    dropout=0.2
)

model = model.to(device)

print(model)

LSTMRegressor(
  (lstm): LSTM(105, 64, num_layers=2, batch_first=True, dropout=0.2)
  (regressor): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [10]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

use_amp = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

print("AMP enabled:", use_amp)

AMP enabled: False


In [11]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    y_mean,
    y_std,
    scaler,
    use_amp
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        # Standardize target
        y_scaled = (
            (y_batch.numpy() - y_mean)
            / y_std
        )

        y_scaled = torch.tensor(
            y_scaled,
            dtype=torch.float32,
            device=device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp
        ):

            predictions = model(
                X_batch
            )

            loss = criterion(
                predictions,
                y_scaled
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = X_batch.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

    return total_loss / total_samples

In [12]:
def evaluate_loss(
    model,
    loader,
    criterion,
    device,
    y_mean,
    y_std
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                device,
                non_blocking=True
            )

            y_scaled = (
                (y_batch.numpy() - y_mean)
                / y_std
            )

            y_scaled = torch.tensor(
                y_scaled,
                dtype=torch.float32,
                device=device
            )

            predictions = model(
                X_batch
            )

            loss = criterion(
                predictions,
                y_scaled
            )

            batch_size = X_batch.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

    return total_loss / total_samples

In [13]:
EPOCHS = 25
PATIENCE = 5

best_val_loss = float("inf")
patience_counter = 0

history = []

best_model_path = "best_lstm_rul.pt"

print("Starting LSTM training...\n")

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        y_mean,
        y_std,
        scaler,
        use_amp
    )

    val_loss = evaluate_loss(
        model,
        val_loader,
        criterion,
        device,
        y_mean,
        y_std
    )

    elapsed = time.time() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f} | "
        f"Time: {elapsed:.1f}s"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("  → Best model saved.")

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print(
                "\nEarly stopping triggered."
            )

            break

total_time = time.time() - start_time

print(
    f"\nTraining completed in "
    f"{total_time / 60:.2f} minutes."
)

Starting LSTM training...

Epoch 01 | Train Loss: 0.54637 | Val Loss: 0.72688 | Time: 9.4s
  → Best model saved.
Epoch 02 | Train Loss: 0.32207 | Val Loss: 0.84407 | Time: 7.2s
Epoch 03 | Train Loss: 0.23179 | Val Loss: 0.92350 | Time: 8.3s
Epoch 04 | Train Loss: 0.17848 | Val Loss: 0.97720 | Time: 7.1s
Epoch 05 | Train Loss: 0.14316 | Val Loss: 0.91655 | Time: 6.9s
Epoch 06 | Train Loss: 0.11814 | Val Loss: 0.95103 | Time: 6.7s

Early stopping triggered.

Training completed in 0.76 minutes.


In [14]:
model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Best LSTM model loaded.")

Best LSTM model loaded.


In [15]:
def predict_model(
    model,
    loader,
    device,
    y_mean,
    y_std
):

    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                device,
                non_blocking=True
            )

            pred_scaled = model(
                X_batch
            )

            # Convert prediction back to original RUL scale
            pred = (
                pred_scaled.cpu().numpy()
                * y_std
                + y_mean
            )

            predictions.extend(pred)
            actuals.extend(
                y_batch.numpy()
            )

    return (
        np.array(actuals),
        np.array(predictions)
    )

In [16]:
y_test_actual, y_test_pred = predict_model(
    model,
    test_loader,
    device,
    y_mean,
    y_std
)

print("Actual shape:", y_test_actual.shape)
print("Prediction shape:", y_test_pred.shape)

Actual shape: (10075,)
Prediction shape: (10075,)


In [17]:
lstm_mae = mean_absolute_error(
    y_test_actual,
    y_test_pred
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        y_test_actual,
        y_test_pred
    )
)

lstm_r2 = r2_score(
    y_test_actual,
    y_test_pred
)

print("===== LSTM TEST RESULTS =====")

print(f"MAE  : {lstm_mae:.4f}")
print(f"RMSE : {lstm_rmse:.4f}")
print(f"R²   : {lstm_r2:.4f}")

===== LSTM TEST RESULTS =====
MAE  : 44.9808
RMSE : 60.6743
R²   : 0.2307


In [18]:
y_val_actual, y_val_pred = predict_model(
    model,
    val_loader,
    device,
    y_mean,
    y_std
)

val_mae = mean_absolute_error(
    y_val_actual,
    y_val_pred
)

val_rmse = np.sqrt(
    mean_squared_error(
        y_val_actual,
        y_val_pred
    )
)

val_r2 = r2_score(
    y_val_actual,
    y_val_pred
)

print("===== LSTM VALIDATION RESULTS =====")

print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

===== LSTM VALIDATION RESULTS =====
MAE  : 46.9072
RMSE : 59.7202
R²   : 0.2450


In [19]:
baseline_results = pd.DataFrame({
    "Model": [
        "Mean RUL",
        "Linear Regression",
        "Random Forest",
        "LSTM"
    ],

    "MAE": [
        55.3885,
        51.8206,
        43.2011,
        lstm_mae
    ],

    "RMSE": [
        69.1759,
        68.6914,
        56.2297,
        lstm_rmse
    ],

    "R2": [
        -0.0001,
        0.0139,
        0.3392,
        lstm_r2
    ]
})

display(
    baseline_results.round(4)
)

,Model,MAE,RMSE,R2
0,Mean RUL,55.3885,69.1759,-0.0001
1,Linear Regression,51.8206,68.6914,0.0139
2,Random Forest,43.2011,56.2297,0.3392
3,LSTM,44.9808,60.6743,0.2307


In [20]:
baseline_results.to_csv(
    "lstm_vs_baselines_results.csv",
    index=False
)

pd.DataFrame(history).to_csv(
    "lstm_training_history.csv",
    index=False
)

print("Saved:")
print("1. lstm_vs_baselines_results.csv")
print("2. lstm_training_history.csv")
print("3. best_lstm_rul.pt")

Saved:
1. lstm_vs_baselines_results.csv
2. lstm_training_history.csv
3. best_lstm_rul.pt


In [21]:
print("===== PREDICTION SANITY CHECK =====")

print(
    "Actual RUL range:",
    y_test_actual.min(),
    "to",
    y_test_actual.max()
)

print(
    "Predicted RUL range:",
    y_test_pred.min(),
    "to",
    y_test_pred.max()
)

print(
    "Mean actual RUL:",
    y_test_actual.mean()
)

print(
    "Mean predicted RUL:",
    y_test_pred.mean()
)

===== PREDICTION SANITY CHECK =====
Actual RUL range: 0.2 to 384.4
Predicted RUL range: 16.19246 to 346.07574
Mean actual RUL: 88.839386
Mean predicted RUL: 85.869514
